# 02 · SmolVLA inference (GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rsasaki0109/vla_ros2/blob/main/notebooks/02_smolvla_inference.ipynb)

Runtime → Change runtime type → **T4 GPU**. Loads `lerobot/smolvla_base` through the
`vla_ros2` adapter, measures latency, and records the closed-loop demo on the SO-100
kinematic stand-in (not a physics sim, not a success benchmark).

`smolvla_base` outputs actions in its own normalized space with generic camera names;
it is a starting point for fine-tuning, not a ready SO-100 policy (see notebook 03).

In [ ]:
# Colab: clone the repo. Locally: run from the repo (or its notebooks/ folder).
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("vla_ros2").exists():
    !git clone --depth 1 https://github.com/rsasaki0109/vla_ros2.git
REPO = Path("vla_ros2").resolve() if IN_COLAB else Path.cwd()
if not (REPO / "pyproject.toml").exists():
    REPO = REPO.parent
os.chdir(REPO)
print("repo:", REPO)

In [ ]:
%pip install -q -e ".[smolvla]"

In [ ]:
import torch

print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## Load and time one action chunk

In [ ]:
import time

import numpy as np

from vla_ros2 import load_model
from vla_ros2.core.types import VLAObservation

start = time.perf_counter()
model = load_model("smolvla", pretrained="lerobot/smolvla_base", device=DEVICE, return_action_chunk=True)
print(f"load {time.perf_counter() - start:.1f}s", model.metadata["image_keys"])

rng = np.random.default_rng(0)
observation = VLAObservation(
    instruction="Put the red cube on top of the blue cube.",
    images={"primary": rng.integers(0, 255, (256, 256, 3), dtype=np.uint8)},
    state={"state": [0.0, -90.0, 75.0, 70.0, 80.0, 0.5]},
)
latencies = []
for _ in range(6):
    start = time.perf_counter()
    chunk = model.predict_observation(observation)
    latencies.append((time.perf_counter() - start) * 1000)
print(f"chunk of {len(chunk.actions)} actions; latency ms (first is warm-up):", [round(x) for x in latencies])
print("first action:", np.round(chunk.actions[0].data, 3))

## Closed-loop demo on the SO-100 kinematic stand-in

In [ ]:
!{sys.executable} scripts/record_smolvla_so100_demo.py --device {DEVICE} --steps 40 --out /tmp/smolvla_demo.gif 2>&1 | tail -3
from IPython.display import Image as GifImage

GifImage(filename="/tmp/smolvla_demo.gif")